# PROJECT : 1

## House Price Predictor

### Phase:1 DATA LOADING & EDA

In [1]:
import pandas as pd
import numpy as np

In [2]:
from sklearn.datasets import fetch_california_housing
housing = fetch_california_housing(as_frame=True)

df = housing.frame

print("Shape:", df.shape)
print("\nFeatures:")
print(df.columns)

print("\nHead:")
print(df.head())

print("\nDescribe:")
print(df.describe())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicates:", df.duplicated().sum())

target = df["MedHouseVal"]

print("\nTarget Statistics")
print("Mean:", target.mean())
print("Median:", target.median())
print("Std:", target.std())
print("Skew:", target.skew())
print("Min:", target.min())
print("Max:", target.max())

corr = df.corr()["MedHouseVal"].sort_values(
    key=abs,
    ascending=False
)

print("\nCorrelation with Target")
print(corr)

Shape: (20640, 9)

Features:
Index(['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup',
       'Latitude', 'Longitude', 'MedHouseVal'],
      dtype='object')

Head:
   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  MedHouseVal  
0    -122.23        4.526  
1    -122.22        3.585  
2    -122.24        3.521  
3    -122.25        3.413  
4    -122.25        3.422  

Describe:
             MedInc      HouseAge      AveRooms     AveBedrms    Population  \
count  20640.000000  20640.000000  20640.000000  20640.000000  20640.000000   
mean       3.870671 

### Phase:2 FEATURE ENGINEERING

In [3]:
df["rooms_per_household"] = df["AveRooms"] / df["AveOccup"]

df["bedrooms_ratio"] = df["AveBedrms"] / df["AveRooms"]

df["population_per_sqmile"] = df["Population"] / df["AveOccup"]

df["income_per_room"] = df["MedInc"] / (df["AveRooms"] + 0.001)

df["coastal_distance"] = (
    abs(df["Latitude"] - 36)
    + abs(df["Longitude"] + 120)
)

df["age_category"] = pd.cut(
    df["HouseAge"],
    bins=3,
    labels=["New", "Mid", "Old"]
)

df["high_income_flag"] = (
    df["MedInc"] >
    df["MedInc"].quantile(0.75)
).astype(int)

# Convert category to numbers
df["age_category"] = df["age_category"].cat.codes

corr2 = df.corr()["MedHouseVal"].sort_values(
    key=abs,
    ascending=False
)

print(corr2.head(10))

MedHouseVal            1.000000
MedInc                 0.688075
income_per_room        0.665326
high_income_flag       0.549872
bedrooms_ratio        -0.255624
rooms_per_household    0.209482
AveRooms               0.151948
Latitude              -0.144160
HouseAge               0.105623
age_category           0.086797
Name: MedHouseVal, dtype: float64


### Phase:3 OUTLIERS & SPLIT

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
numeric_cols = df.select_dtypes(include=np.number).columns

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = ((df[col] < lower) |
                (df[col] > upper)).sum()

    print(col, ":", outliers)

X = df.drop("MedHouseVal", axis=1)
y = df["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

MedInc : 681
HouseAge : 0
AveRooms : 511
AveBedrms : 1424
Population : 1196
AveOccup : 711
Latitude : 0
Longitude : 0
MedHouseVal : 1071
rooms_per_household : 402
bedrooms_ratio : 591
population_per_sqmile : 1220
income_per_room : 349
coastal_distance : 3784
age_category : 0
high_income_flag : 5160


### Phase:4 MODEL TRAINING

In [5]:
import time
import pandas as pd

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score

models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(),
    "Random Forest": RandomForestRegressor(
        n_estimators=50,
        random_state=42
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42
    )
}

results = []

# Train and compare models

for name, model in models.items():

    print(f"\nTraining {name} ...")

    start = time.time()

    scores = cross_val_score(
        model,
        X_train_scaled,
        y_train,
        cv=5,
        scoring="neg_root_mean_squared_error"
    )

    rmse = -scores.mean()
    std = scores.std()

    elapsed = time.time() - start

    results.append([
        name,
        round(rmse, 4),
        round(std, 4),
        round(elapsed, 2)
    ])

    print(f"{name} Completed")

# Results table

result_df = pd.DataFrame(
    results,
    columns=["Model", "RMSE", "STD", "Time(sec)"]
)

print("\nModel Comparison")
print(result_df)

# Best model

best_model_name = result_df.sort_values(
    by="RMSE"
).iloc[0]["Model"]

print("\nBest Model:", best_model_name)


Training Linear Regression ...
Linear Regression Completed

Training Ridge ...
Ridge Completed

Training Random Forest ...
Random Forest Completed

Training Gradient Boosting ...
Gradient Boosting Completed

Model Comparison
               Model    RMSE     STD  Time(sec)
0  Linear Regression  0.6723  0.0226       0.05
1              Ridge  0.6723  0.0226       0.03
2      Random Forest  0.5025  0.0021      80.59
3  Gradient Boosting  0.5334  0.0062      44.49

Best Model: Random Forest


### Phase:5 HYPERPARAMETER TUNING

In [6]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor

print("Starting Hyperparameter Tuning...")

# Parameter grid
param_grid = {
    "n_estimators": [50, 100, 150],
    "max_depth": [5, 10, 15, None],
    "min_samples_split": [2, 5, 10]
}

# Random Search
search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_distributions=param_grid,
    n_iter=5,          # small for faster execution
    cv=3,              # 3-fold CV
    scoring="neg_root_mean_squared_error",
    random_state=42,
    verbose=2,
    n_jobs=-1
)

# Train
search.fit(X_train_scaled, y_train)

print("\n==============================")
print("BEST PARAMETERS")
print("==============================")
print(search.best_params_)

print("\nBest CV Score (RMSE):")
print(-search.best_score_)

# Best model
best_model = search.best_estimator_

print("\nFinal Tuned Model:")
print(best_model)

print("\nHyperparameter Tuning Complete!")

Starting Hyperparameter Tuning...
Fitting 3 folds for each of 5 candidates, totalling 15 fits

BEST PARAMETERS
{'n_estimators': 150, 'min_samples_split': 10, 'max_depth': None}

Best CV Score (RMSE):
0.5087559218506242

Final Tuned Model:
RandomForestRegressor(min_samples_split=10, n_estimators=150, random_state=42)

Hyperparameter Tuning Complete!


### Phase:6 EVALUATION

In [7]:
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)
pred = best_model.predict(X_test_scaled)

rmse = np.sqrt(
    mean_squared_error(y_test, pred)
)

mae = mean_absolute_error(
    y_test,
    pred
)

r2 = r2_score(
    y_test,
    pred
)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

residuals = y_test - pred

print("\nResidual Analysis")
print("Mean:", residuals.mean())
print("Std:", residuals.std())
print("Min:", residuals.min())
print("Max:", residuals.max())

importances = pd.Series(
    best_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print("\nTop 10 Features")
print(importances.head(10))

RMSE: 0.4917143079797953
MAE: 0.32092465391728525
R2: 0.815490457798435

Residual Analysis
Mean: -0.008782955264453553
Std: 0.49169542129570926
Min: -2.898981790954048
Max: 3.3674795250708334

Top 10 Features
income_per_room        0.359765
MedInc                 0.168202
rooms_per_household    0.140513
Latitude               0.060288
Longitude              0.059420
AveOccup               0.055766
coastal_distance       0.048137
HouseAge               0.035460
bedrooms_ratio         0.017695
AveBedrms              0.017403
dtype: float64


### Phase:7 AVM CLASS

In [8]:
import time
class HousePriceAVM:

    def __init__(self, model, scaler):
        self.model = model
        self.scaler = scaler

    def predict(self, features):

        df_new = pd.DataFrame([features])

        df_scaled = self.scaler.transform(df_new)

        prediction = self.model.predict(df_scaled)[0]

        lower = prediction * 0.9
        upper = prediction * 1.1

        return prediction, (lower, upper)

    def evaluate(self, X, y):

        pred = self.model.predict(
            self.scaler.transform(X)
        )

        return {
            "RMSE": np.sqrt(
                mean_squared_error(y, pred)
            ),
            "MAE": mean_absolute_error(y, pred),
            "R2": r2_score(y, pred)
        }

    def batch_predict(self, df):

        scaled = self.scaler.transform(df)

        return self.model.predict(scaled)

avm = HousePriceAVM(
    best_model,
    scaler
)

print("AVM Ready")

AVM Ready


### Phase:8 FINAL REPORT

In [9]:
print("="*60)

print("HOUSE PRICE PREDICTOR REPORT")

print("\n1. Business Problem")
print("Build an Automated Valuation Model (AVM)")
print("to estimate house prices accurately.")

print("\n2. Best Model")
print(best_model)

print("\n3. Important Features")
print(importances.head(10))

print("\n4. Business Impact")
print(
    "Better pricing reduces overpricing and "
    "underpricing of properties."
)

print("\n5. Production Checklist")
checklist = [
    "Data Validation",
    "Feature Engineering",
    "Scaling",
    "Model Versioning",
    "Monitoring",
    "Retraining",
    "Logging",
    "API Deployment",
    "Error Handling",
    "Security",
    "Testing",
    "Documentation"
]

for item in checklist:
    print("-", item)

print("\n6. Future Improvements")
print("- Satellite Images")
print("- School Ratings")
print("- Transport Access")
print("- Crime Statistics")
print("- Nearby Amenities")

print("="*60)

HOUSE PRICE PREDICTOR REPORT

1. Business Problem
Build an Automated Valuation Model (AVM)
to estimate house prices accurately.

2. Best Model
RandomForestRegressor(min_samples_split=10, n_estimators=150, random_state=42)

3. Important Features
income_per_room        0.359765
MedInc                 0.168202
rooms_per_household    0.140513
Latitude               0.060288
Longitude              0.059420
AveOccup               0.055766
coastal_distance       0.048137
HouseAge               0.035460
bedrooms_ratio         0.017695
AveBedrms              0.017403
dtype: float64

4. Business Impact
Better pricing reduces overpricing and underpricing of properties.

5. Production Checklist
- Data Validation
- Feature Engineering
- Scaling
- Model Versioning
- Monitoring
- Retraining
- Logging
- API Deployment
- Error Handling
- Security
- Testing
- Documentation

6. Future Improvements
- Satellite Images
- School Ratings
- Transport Access
- Crime Statistics
- Nearby Amenities
